# Notebook 05 – Visualization & Final Interpretation
## Echo Chambers in Climate Change Discussions on Reddit

Produces all **publication-quality figures** for the final report and generates
the interpretation/discussion section.

### Figures produced:
1. Network snapshots (community-colored) for 3 key periods
2. Community evolution heatmap
3. Final combined dashboard (all metrics)
4. Gephi export files for beautiful network renders

In [ ]:
import sys, pickle
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import networkx as nx
from pathlib import Path

from src.metrics import GraphMetrics
from src.visualizer import Visualizer
from src.network_builder import CLIMATE_EVENTS

GRAPH_DIR   = Path('../data/processed/graphs')
FIGURES_DIR = Path('../figures')
REPORT_DIR  = Path('../report')
GEPHI_DIR   = Path('../figures/gephi_exports')
GEPHI_DIR.mkdir(parents=True, exist_ok=True)

viz = Visualizer(output_dir=str(FIGURES_DIR))
gm  = GraphMetrics(betweenness_k=100)

print("✅ Ready")

## Step 1: Load All Saved Data

In [ ]:
with open(GRAPH_DIR / 'monthly_graphs.pkl', 'rb') as f:
    monthly_graphs = pickle.load(f)

with open(GRAPH_DIR / 'event_graphs.pkl', 'rb') as f:
    event_graphs = pickle.load(f)

temporal_df  = pd.read_csv(REPORT_DIR / 'temporal_metrics.csv', index_col=0)
event_df     = pd.read_csv(REPORT_DIR / 'event_comparison.csv', index_col=0)
bridge_df    = pd.read_csv(REPORT_DIR / f'bridge_users_{sorted(monthly_graphs.keys())[-1]}.csv')
persistent_df = pd.read_csv(REPORT_DIR / 'persistent_bridge_users.csv')

print(f"Periods    : {len(temporal_df)}")
print(f"Events     : {len(event_df)}")
print(f"Bridge users: {len(bridge_df)}")
print(f"Persistent bridges: {len(persistent_df)}")

## Step 2: Network Snapshots (3 Key Periods)

In [ ]:
# Pick early, middle, and late periods for visual comparison
sorted_periods = sorted(monthly_graphs.keys())
n = len(sorted_periods)

snapshot_labels = [
    sorted_periods[0],           # earliest
    sorted_periods[n // 2],      # middle
    sorted_periods[-1],          # latest
]
print(f"Snapshot periods: {snapshot_labels}")

In [ ]:
# Generate a community-colored snapshot for each period
for label in snapshot_labels:
    G = monthly_graphs[label]
    stats = gm.compute_all(G, label=label)
    partition = stats.get('_partition', {})
    
    print(f"\nGenerating snapshot: {label}  "
          f"({G.number_of_nodes()} nodes, modularity={stats['modularity']:.3f})")
    
    viz.plot_network_snapshot(
        G, partition, label=label,
        max_nodes=400,
        filename=f'network_snapshot_{label}.png'
    )

## Step 3: Community Evolution Heatmap

In [ ]:
# Collect partitions for all monthly graphs
partitions = {}
for label, G in monthly_graphs.items():
    partition, _ = gm.detect_communities(G)
    partitions[label] = partition

viz.plot_community_heatmap(
    monthly_graphs, partitions,
    top_communities=10,
    filename='community_heatmap.png'
)
plt.show()
print("✅ Community heatmap saved")

## Step 4: Bridge Users (All Periods)

In [ ]:
viz.plot_bridge_users(bridge_df, top_n=20, filename='bridge_users_top20.png')
plt.show()

# Persistent bridge users across all time
viz.plot_bridge_users(
    persistent_df.rename(columns={'avg_betweenness': 'betweenness'}),
    top_n=15, filename='persistent_bridges.png',
    title='Most Persistent Bridge Users (≥3 Months Active)'
)
plt.show()

## Step 5: Final Dashboard (All Key Metrics)

In [ ]:
fig = plt.figure(figsize=(20, 14), facecolor='#0f1117')
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.4)

ax_mod  = fig.add_subplot(gs[0, :2])
ax_ass  = fig.add_subplot(gs[1, :2])
ax_com  = fig.add_subplot(gs[2, :2])
ax_pie  = fig.add_subplot(gs[0, 2])
ax_box  = fig.add_subplot(gs[1, 2])
ax_br   = fig.add_subplot(gs[2, 2])

x = range(len(temporal_df))
labels = list(temporal_df.index)
step = max(1, len(labels) // 8)

DARK_AX = {'facecolor': '#1a1d27', 'spines': '#444'}

def style_ax(ax):
    ax.set_facecolor('#1a1d27')
    for sp in ax.spines.values(): sp.set_color('#444')
    ax.tick_params(colors='#aaa', labelsize=7)
    ax.grid(color='#2d2d3a', linestyle='--', alpha=0.5)

# Modularity
y_mod = temporal_df['modularity'].values
ax_mod.plot(x, y_mod, color='#00e5ff', lw=2)
ax_mod.fill_between(x, y_mod, alpha=0.15, color='#00e5ff')
ax_mod.set_title('Modularity Over Time', color='white', fontsize=11)
ax_mod.set_xticks(range(0, len(labels), step))
ax_mod.set_xticklabels(labels[::step], rotation=30, ha='right')
style_ax(ax_mod)

# Assortativity
y_ass = temporal_df['degree_assortativity'].values
ax_ass.plot(x, y_ass, color='#ff6b6b', lw=2)
ax_ass.fill_between(x, y_ass, alpha=0.15, color='#ff6b6b')
ax_ass.axhline(0, color='#aaa', lw=0.8)
ax_ass.set_title('Degree Assortativity Over Time', color='white', fontsize=11)
ax_ass.set_xticks(range(0, len(labels), step))
ax_ass.set_xticklabels(labels[::step], rotation=30, ha='right')
style_ax(ax_ass)

# Communities
y_com = temporal_df['num_communities'].values
ax_com.bar(x, y_com, color='#51cf66', alpha=0.8)
ax_com.set_title('Number of Communities Over Time', color='white', fontsize=11)
ax_com.set_xticks(range(0, len(labels), step))
ax_com.set_xticklabels(labels[::step], rotation=30, ha='right')
style_ax(ax_com)

# Pie: avg modularity before vs after events
if not event_df.empty and 'before_modularity' in event_df.columns:
    avg_before = event_df['before_modularity'].mean()
    avg_after  = event_df['after_modularity'].mean()
    ax_pie.pie([avg_before, avg_after], labels=['Avg Before\nEvents', 'Avg After\nEvents'],
               colors=['#ff6b6b', '#51cf66'], autopct='%1.2f',
               textprops={'color': 'white', 'fontsize': 8}, startangle=90)
    ax_pie.set_title('Avg Modularity\nBefore vs After Events', color='white', fontsize=10)
    ax_pie.set_facecolor('#1a1d27')

# Box: modularity distribution
bp = ax_box.boxplot(y_mod[~np.isnan(y_mod)], vert=True,
                    patch_artist=True, widths=0.6,
                    medianprops={'color': '#ffd43b', 'linewidth': 2})
for patch in bp['boxes']: patch.set_facecolor('#00e5ff'); patch.set_alpha(0.6)
ax_box.set_title('Modularity Distribution', color='white', fontsize=10)
ax_box.set_xticklabels(['Modularity'])
style_ax(ax_box)

# Top bridge users
if not bridge_df.empty:
    top_b = bridge_df.head(8).sort_values('betweenness', ascending=True)
    ax_br.barh(top_b['user'], top_b['betweenness'], color='#a855f7', alpha=0.85)
    ax_br.set_title('Top Bridge Users', color='white', fontsize=10)
    ax_br.set_xlabel('Betweenness', color='#aaa', fontsize=8)
    style_ax(ax_br)

fig.suptitle(
    'Echo Chamber Analysis – Climate Subreddits\nSocial Network Analysis Dashboard',
    fontsize=16, fontweight='bold', color='white', y=1.01
)

plt.savefig(str(FIGURES_DIR / 'final_dashboard.png'), dpi=150,
            bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("✅ Dashboard saved")

## Step 6: Export to Gephi Format (GraphML)

In [ ]:
# Export 3 key snapshots to GraphML for Gephi
sorted_periods = sorted(monthly_graphs.keys())
n = len(sorted_periods)
gephi_periods = [sorted_periods[0], sorted_periods[n//2], sorted_periods[-1]]

for label in gephi_periods:
    G = monthly_graphs[label]
    
    # Add community attribute for Gephi coloring
    partition, modularity = gm.detect_communities(G)
    for node in G.nodes():
        G.nodes[node]['community'] = partition.get(node, -1)
    
    # Convert degree dict to node attribute
    for node, deg in G.degree():
        G.nodes[node]['degree'] = deg
    
    safe_label = label.replace('-', '_')
    out_path = GEPHI_DIR / f'network_{safe_label}.graphml'
    nx.write_graphml(G, str(out_path))
    print(f"✅ Gephi export: {out_path}  (modularity={modularity:.4f})")

## Step 7: Interpretation & Discussion

Run this cell to auto-generate an interpretation based on your actual results:

In [ ]:
mod_vals = temporal_df['modularity'].dropna()
ass_vals = temporal_df['degree_assortativity'].dropna()
com_vals = temporal_df['num_communities'].dropna()

# Trend direction (simple linear regression slope)
from scipy import stats as spstats
x_idx = np.arange(len(mod_vals))

mod_slope, _, mod_r, mod_p, _ = spstats.linregress(x_idx, mod_vals)
ass_slope, _, ass_r, ass_p, _ = spstats.linregress(x_idx, ass_vals)

# Event impact: average delta modularity across events
if 'delta_modularity' in event_df.columns:
    avg_delta = event_df['delta_modularity'].mean()
else:
    avg_delta = 0.0

direction_mod = '📈 increasing' if mod_slope > 0 else '📉 decreasing'
direction_ass = '📈 increasing' if ass_slope > 0 else '📉 decreasing'
event_effect  = 'strengthening' if avg_delta > 0 else 'weakening'

interpretation = f"""
╔══════════════════════════════════════════════════════════════════════╗
║        INTERPRETATION & FINDINGS                                     ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  MODULARITY TREND : {direction_mod}                                  ║
║    • Mean: {mod_vals.mean():.4f}  |  Slope: {mod_slope:+.5f}  |  R²: {mod_r**2:.3f}    ║
║    • {'Echo chambers appear to be growing stronger over time.' if mod_slope > 0 else 'Echo chambers appear to be weakening — communities are mixing.'}
║                                                                      ║
║  ASSORTATIVITY TREND : {direction_ass}                               ║
║    • Mean: {ass_vals.mean():.4f}  |  Slope: {ass_slope:+.5f}  |  R²: {ass_r**2:.3f}   ║
║    • {'Users prefer connecting with similarly-connected peers (homophily).' if ass_vals.mean() > 0 else 'Network shows disassortative mixing — hubs connect to low-degree nodes.'}
║                                                                      ║
║  CLIMATE EVENTS EFFECT :                                             ║
║    • Avg. Δ Modularity (after − before) = {avg_delta:+.4f}           ║
║    • Events are {event_effect} community separation.                 ║
║                                                                      ║
║  COMMUNITIES :                                                       ║
║    • Avg. communities per month: {com_vals.mean():.1f}               ║
║    • Range: {int(com_vals.min())} – {int(com_vals.max())}            ║
║                                                                      ║
║  BRIDGE USERS :                                                      ║
║    • {len(persistent_df)} users consistently bridge communities across multiple periods.  ║
╚══════════════════════════════════════════════════════════════════════╝
"""

print(interpretation)

# Save to report
with open(REPORT_DIR / 'interpretation.txt', 'w') as f:
    f.write(interpretation)
print(f"✅ Interpretation saved → {REPORT_DIR / 'interpretation.txt'}")

## ✅ Project Complete!

### All outputs are in:
- `figures/` — all PNG plots
- `figures/gephi_exports/` — GraphML files for Gephi
- `report/` — CSV tables + interpretation text

### Next steps:
1. Open `.graphml` files in **Gephi** for beautiful network renders
2. Paste the interpretation and figures into your final report
3. Compare your findings with Treen et al. (2022) and discuss differences